# InsertionSort (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness/InsertionSort_cpp.ipynb)

Chapter 13 — Correctness. Runnable notebook of `InsertionSort`: the algorithm in C++, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/analysis/13-correctness/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness/InsertionSort_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness/InsertionSort_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Indices.** In the pseudo-code the arrays are indexed from 1; in the C++ code from 0: the element `a[i]` of the pseudo-code is `a[i-1]` in the code (and a loop `while i <= n` from 1 becomes `while i < n` from 0). The printed tables use the indices of the pseudo-code.

## Insertion sort: InsertionSort

**Input:** an array $a[1..n]$ of $n \ge 1$ natural numbers. **Output:** the array sorted in non-decreasing order. **Outer invariant:** $a[1..i-1]$ contains the original elements of $a[1..i-1]$ sorted and $a[i..n]$ equals the original. **Inner invariant** (I1)–(I3): $a[1..j] = b[1..j]$, $a[j+2..i] = b[j+1..i-1]$ and these elements are $> key$.

Pseudo-code: section 13.3 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis/13-correctness/).

In [1]:
%%writefile insertion_sort.cpp
#include <algorithm>
#include <cstdio>
#include <string>
using namespace std;

const int MAXN = 64;

// "[a[from], ..., a[to-1]]"
string Fmt(const int* a, int from, int to)
{
    string s = "[";
    for (int i = from; i < to; i++)
    {
        if (i > from) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

bool Equal(const int* x, const int* y, int len)     // x[0..len-1] == y[0..len-1]
{
    for (int k = 0; k < len; k++)
        if (x[k] != y[k]) return false;
    return true;
}

// The array with a bar after the sorted part a[1..i-1]
string Split(const int* a, int n, int i)
{
    string s = Fmt(a, 0, i);
    s.pop_back();
    string t = Fmt(a, i, n);
    return s + " | " + t.substr(1);
}

// Outer invariant: a[1..i-1] = original elements of a[1..i-1] sorted, a[i..n] = original
void PrintOuter(const int* a, const int* orig, int n, int i)
{
    int s[MAXN];
    for (int k = 0; k < i; k++) s[k] = orig[k];
    sort(s, s + i);
    bool ok = Equal(a, s, i) && Equal(a + i, orig + i, n - i);
    printf("%2d | %-26s | %-19s | %s\n", i + 1, Split(a, n, i).c_str(), Fmt(orig, 0, i).c_str(), ok ? "ok" : "FALSE");
}

// Inner invariant (I1)-(I3); b = a at the start of the iteration
void PrintInner(const int* a, const int* b, int n, int i, int j, int key)
{
    int ip = i + 1, jp = j + 1;                    // indices of the pseudo-code
    bool ok = Equal(a, b, jp) && Equal(a + jp + 1, b + jp, ip - jp - 1);
    for (int k = jp + 1; k < ip; k++)
        if (a[k] <= key) ok = false;
    string cond = "j = 0: false";
    if (j >= 0)
    {
        bool t = a[j] > key;
        cond = "a[" + to_string(jp) + "] = " + to_string(a[j]) + (t ? " > " : " <= ") + to_string(key) + ": " + (t ? "true" : "false");
    }
    printf("%d | %-15s | %-12s | %-9s | %-20s | %s\n", jp, Fmt(a, 0, n).c_str(), Fmt(a, 0, jp).c_str(), Fmt(a, jp + 1, ip).c_str(), cond.c_str(), ok ? "ok" : "FALSE");
}

// InsertionSort(a, n)
void InsertionSort(int a[], int n, bool showOuter = true, int traceInnerAt = 0)
{
    int orig[MAXN], b[MAXN];
    for (int k = 0; k < n; k++) orig[k] = a[k];    // copy of the input: it is only needed to check the invariant
    int i = 1;                                     // i := 2
    if (showOuter) PrintOuter(a, orig, n, i);
    while (i < n)                                  // i <= n
    {
        for (int k = 0; k < n; k++) b[k] = a[k];   // content of a at the start of the iteration (only for the inner invariant)
        int key = a[i];
        int j = i - 1;
        bool tr = (i + 1 == traceInnerAt);
        if (tr) PrintInner(a, b, n, i, j, key);
        while (j >= 0 && a[j] > key)               // j >= 1 and a[j] > key
        {
            a[j + 1] = a[j];
            j = j - 1;
            if (tr) PrintInner(a, b, n, i, j, key);
        }
        a[j + 1] = key;
        i = i + 1;
        if (showOuter) PrintOuter(a, orig, n, i);
    }
}

#ifndef SOLO_LIBRERIA
int main()
{
    // The outer loop: a[1..5] = (5, 2, 4, 1, 3), example of the lecture notes
    int a[] = {5, 2, 4, 1, 3};
    int n = 5;
    printf("a = %s\n", Fmt(a, 0, n).c_str());
    printf(" i | a (sorted | rest)          | original a[1..i-1]  | invariant\n");
    InsertionSort(a, n);
    printf("sorted: %s\n", Fmt(a, 0, n).c_str());

    // The inner loop at iteration i = 5, with key = 3 (a = (1, 2, 4, 5, 3) at the start)
    printf("\n");
    int c[] = {5, 2, 4, 1, 3};
    printf("j | a               | a[1..j]      | a[j+2..i] | condition            | invariant\n");
    InsertionSort(c, n, false, 5);
    printf("after line 10 (a[j+1] = key): a = %s\n", Fmt(c, 0, n).c_str());
    return 0;
}
#endif

Writing insertion_sort.cpp


In [2]:
!g++ -std=c++17 -O2 -o insertion_sort insertion_sort.cpp && ./insertion_sort

a = [5, 2, 4, 1, 3]
 i | a (sorted | rest)          | original a[1..i-1]  | invariant
 2 | [5 | 2, 4, 1, 3]           | [5]                 | ok
 3 | [2, 5 | 4, 1, 3]           | [5, 2]              | ok
 4 | [2, 4, 5 | 1, 3]           | [5, 2, 4]           | ok
 5 | [1, 2, 4, 5 | 3]           | [5, 2, 4, 1]        | ok
 6 | [1, 2, 3, 4, 5 | ]         | [5, 2, 4, 1, 3]     | ok
sorted: [1, 2, 3, 4, 5]

j | a               | a[1..j]      | a[j+2..i] | condition            | invariant
4 | [1, 2, 4, 5, 3] | [1, 2, 4, 5] | []        | a[4] = 5 > 3: true   | ok
3 | [1, 2, 4, 5, 5] | [1, 2, 4]    | [5]       | a[3] = 4 > 3: true   | ok
2 | [1, 2, 4, 4, 5] | [1, 2]       | [4, 5]    | a[2] = 2 <= 3: false | ok
after line 10 (a[j+1] = key): a = [1, 2, 3, 4, 5]


**Try it yourself.** Change the array: every row of the outer table must stay `ok`.

In [3]:
%%writefile prova_insertion_sort.cpp
#define SOLO_LIBRERIA   // do not run the main of insertion_sort.cpp
#include "insertion_sort.cpp"

int main()
{
    int a[] = {6, 3, 8, 1, 9, 2};      // <-- change the array
    int n = 6;
    printf(" i | a (sorted | rest)          | original a[1..i-1]  | invariant\n");
    InsertionSort(a, n);
    return 0;
}

Writing prova_insertion_sort.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_insertion_sort prova_insertion_sort.cpp && ./prova_insertion_sort

 i | a (sorted | rest)          | original a[1..i-1]  | invariant
 2 | [6 | 3, 8, 1, 9, 2]        | [6]                 | ok
 3 | [3, 6 | 8, 1, 9, 2]        | [6, 3]              | ok
 4 | [3, 6, 8 | 1, 9, 2]        | [6, 3, 8]           | ok
 5 | [1, 3, 6, 8 | 9, 2]        | [6, 3, 8, 1]        | ok
 6 | [1, 3, 6, 8, 9 | 2]        | [6, 3, 8, 1, 9]     | ok
 7 | [1, 2, 3, 6, 8, 9 | ]      | [6, 3, 8, 1, 9, 2]  | ok


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).